# DMS Queries Examples

This notebook contains canonical, production-oriented DMS retrieval patterns.
Each section keeps one primary runnable example to make the notebook concise and repeatable.

## Flow
1. **Search basics** - Run fast full-text lookup on asset names (`instances.search`) to get relevant entry points.
2. **Constrained search** - Combine text query with hard filters (for example `sourceContext` and `space`) for precision.
3. **Top-K hydrate** - Take the top search hits and fetch related entities (for example time series) via relation-aware filters.
4. **Fallback** - Use a strict query first, then automatically relax constraints when strict search returns zero hits.
5. **Subtree Prefix** - Retrieve a hierarchy slice efficiently using `Prefix` on `path` (preferred over broad path scans).
6. **Graph traversals** - Use `/query` to traverse parent/children and nested direct-relation filters in one server-side step.
7. **Cursor pagination** - Drain large result sets safely with `/sync` cursors in fixed-size pages with retry handling.

In [23]:
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

from dotenv import load_dotenv

from cognite.client import CogniteClient, global_config
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials
from cognite.client.data_classes.data_modeling import (
    EdgeListWithCursor,
    InstanceSort,
    NodeId,
    NodeListWithCursor,
    ViewId,
    DirectRelationReference,
)
from cognite.client.data_classes.data_modeling.query import (
    NodeResultSetExpression,
    Query,
    QueryResult,
    Select,
    SourceSelector,
)
from cognite.client.data_classes.filters import (
    And,
    ContainsAll,
    ContainsAny,
    Equals,
    HasData,
    Nested,
    Prefix,
)
from cognite.client.exceptions import CogniteAPIError
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter

## Basic setup

This section initializes credentials, client configuration, and environment discovery so every later example can run without hardcoded secrets.

In [24]:
def _find_env_file(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        env_path = candidate / ".env"
        if env_path.is_file():
            return env_path
    return None


env_file = _find_env_file()
if env_file is None:
    raise FileNotFoundError("Could not find .env by walking up from cwd")
load_dotenv(env_file, override=False)

project = os.getenv("CDF_PROJECT")
cluster = os.getenv("CDF_CLUSTER")
client_id = os.getenv("CDF_CLIENT_ID") or os.getenv("IDP_CLIENT_ID")
client_secret = os.getenv("CDF_CLIENT_SECRET") or os.getenv("IDP_CLIENT_SECRET")
tenant_id = os.getenv("CDF_TENANT_ID") or os.getenv("IDP_TENANT_ID")
base_url = os.getenv("CDF_BASE_URL") or os.getenv("CDF_URL") or (f"https://{cluster}.cognitedata.com" if cluster else None)

missing = [
    name for name, val in {
        "CDF_PROJECT": project,
        "CDF_CLIENT_ID / IDP_CLIENT_ID": client_id,
        "CDF_CLIENT_SECRET / IDP_CLIENT_SECRET": client_secret,
        "CDF_TENANT_ID / IDP_TENANT_ID": tenant_id,
        "CDF_BASE_URL / CDF_URL / CDF_CLUSTER": base_url,
    }.items() if not val
]
if missing:
    raise RuntimeError(f"Missing env vars: {', '.join(missing)}")

credentials = OAuthClientCredentials(
    token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    client_id=client_id,
    client_secret=client_secret,
    scopes=[f"{base_url}/.default"],
)

global_config.max_retries = 10
global_config.max_retry_backoff = 30
global_config.disable_pypi_version_check = True

client = CogniteClient(ClientConfig(
    client_name="cdf-query-examples-lean",
    project=project,
    credentials=credentials,
    base_url=base_url,
))

client.config.headers = {**(client.config.headers or {}), "cdf-version": "alpha"}
print(f"Loaded credentials from {env_file}")

Loaded credentials from C:\Users\JanIngeBergseth\OneDrive - Cognite AS\Documents\GitHub\library\.env


## Retry and print helpers

These helpers centralize transient-error retries and compact result printing, so each query example stays short and operationally safe.

In [25]:
RETRYABLE_CODES = {408, 425, 429, 500, 502, 503, 504}


def is_retryable_exception(e: BaseException) -> bool:
    return isinstance(e, CogniteAPIError) and e.code in RETRYABLE_CODES


retry_cognite = retry(
    reraise=True,
    stop=stop_after_attempt(5),
    retry=retry_if_exception(is_retryable_exception),
    wait=wait_exponential_jitter(initial=1, max=30, jitter=2),
)


@retry_cognite
def run_query(client: CogniteClient, query: Query) -> QueryResult:
    return client.data_modeling.instances.query(query=query)


@retry_cognite
def run_sync(client: CogniteClient, query: Query) -> QueryResult:
    return client.data_modeling.instances.sync(query=query)


def log_api_error(e: CogniteAPIError) -> None:
    print(f"CogniteAPIError code={e.code} x_request_id={getattr(e, 'x_request_id', None)}: {e.message}")


def _summarize(label: str, nodes, vid: ViewId, max_rows: int = 10, extra_fields: list[str] | None = None) -> None:
    from itertools import islice

    print(f"{label} (count={len(nodes)}):")
    for n in islice(nodes, max_rows):
        props = n.properties.get(vid, {}) or {}
        parts = [
            f"{n.external_id}",
            f"name={props.get('name')!r}",
            f"description={props.get('description')!r}",
        ]
        for key in (extra_fields or []):
            parts.append(f"{key}={props.get(key)!r}")
        print("  " + "  ".join(parts))
    if len(nodes) > max_rows:
        print(f"   ... and {len(nodes) - max_rows} more")

## Model setup

Define instance space and `ViewId`s in one place to keep all query snippets portable across projects and model deployments.

In [26]:
INST_SP = "inst_location"
MODEL_SP_CDM = "cdf_cdm"
MODEL_SP_IDM = "cdf_idm"
MODEL_VERSION = "v1"

asset_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteAsset", version=MODEL_VERSION)
ts_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteTimeSeries", version=MODEL_VERSION)
eq_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteEquipment", version=MODEL_VERSION)
wo_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteMaintenanceOrder", version=MODEL_VERSION)

print("Using views:", asset_vid, ts_vid, eq_vid, wo_vid)

Using views: ViewId(space='cdf_cdm', external_id='CogniteAsset', version='v1') ViewId(space='cdf_cdm', external_id='CogniteTimeSeries', version='v1') ViewId(space='cdf_cdm', external_id='CogniteEquipment', version='v1') ViewId(space='cdf_idm', external_id='CogniteMaintenanceOrder', version='v1')


## 1) Search basics

Start with lightweight full-text lookup to identify relevant anchors before running heavier graph retrieval steps.

In [27]:
sample = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=10)
sample_name = next(
    ((n.properties.get(asset_vid, {}) or {}).get("name") for n in sample if (n.properties.get(asset_vid, {}) or {}).get("name")),
    None,
)
if not sample_name:
    raise RuntimeError("No assets with name found in INST_SP")

search_term = sample_name[:-2] if len(sample_name) > 2 else sample_name
hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=search_term,
    properties=["name"],
    limit=25,
)
print(f"Name-only search for {search_term!r}")
_summarize("hits", hits, asset_vid)

Name-only search for 'Electrical Distributi'
hits (count=3):
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'


## 2) Constrained search (query + hard filter)

Pair text relevance with explicit scope constraints (`space`, `sourceContext`, etc.) to reduce false positives and backend work.

In [28]:
probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
source_context = next(
    ((n.properties.get(asset_vid, {}) or {}).get("sourceContext") for n in probe if (n.properties.get(asset_vid, {}) or {}).get("sourceContext")),
    None,
)
if source_context is None:
    raise RuntimeError("No populated sourceContext found in INST_SP")

token = sample_name[:4] if sample_name else ""
scoped_hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=token,
    properties=["name"],
    filter=Equals(asset_vid.as_property_ref("sourceContext"), source_context),
    limit=25,
)
print(f"Scoped search (query={token!r}, sourceContext={source_context!r})")
_summarize("scoped_hits", scoped_hits, asset_vid)

Scoped search (query='Elec', sourceContext='cfihos_test')
scoped_hits (count=3):
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'


## 3) Top-K hydrate (search anchors -> related data)

Limit expensive relation lookups to the top-ranked search hits instead of traversing from the full dataset.

In [29]:
top_hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=sample_name[:4] if sample_name else "",
    properties=["name"],
    limit=10,
)
_summarize("top_hits", top_hits, asset_vid)

asset_refs = [DirectRelationReference(n.space, n.external_id) for n in top_hits]
if asset_refs:
    ts_related = client.data_modeling.instances.search(
        view=ts_vid,
        space=INST_SP,
        query=None,
        filter=ContainsAny(ts_vid.as_property_ref("assets"), asset_refs),
        limit=100,
    )
    _summarize("related_timeseries", ts_related, ts_vid, extra_fields=["unit"])
else:
    print("No top hits found; skipping related retrieval")

top_hits (count=3):
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
related_timeseries (count=0):


## 4) Fallback (strict -> broad)

Use controlled relaxation to preserve precision when possible while still returning useful results in sparse or noisy datasets.

In [30]:
strict_token = sample_name[:4] if sample_name and len(sample_name) >= 4 else sample_name
strict_hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=strict_token,
    properties=["name"],
    filter=Equals(asset_vid.as_property_ref("sourceContext"), source_context),
    limit=25,
)

if strict_hits:
    print("Strict search succeeded")
    _summarize("strict_hits", strict_hits, asset_vid)
else:
    print("Strict search returned 0 hits; broadening")
    broad_hits = client.data_modeling.instances.search(
        view=asset_vid,
        space=INST_SP,
        query=strict_token,
        properties=["name"],
        limit=25,
    )
    _summarize("broad_hits", broad_hits, asset_vid)

Strict search succeeded
strict_hits (count=3):
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'


## 5) Subtree retrieval with Prefix (recommended)

`Prefix(path)` is usually the most efficient subtree pattern in DMS and should be preferred over broader path membership scans.

In [31]:
probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
sub_tree_root = None
for node in probe:
    parent = (node.properties.get(asset_vid, {}) or {}).get("parent")
    if parent and parent.get("space") and parent.get("externalId"):
        sub_tree_root = NodeId(parent["space"], parent["externalId"])
        break
if sub_tree_root is None:
    raise RuntimeError("No parent relation found to derive subtree root")

root = client.data_modeling.instances.retrieve_nodes(sub_tree_root, sources=asset_vid)
root_path = root.properties.data[asset_vid]["path"]

start_time = time.time()
sub_tree_nodes = client.data_modeling.instances.list(
    sources=asset_vid,
    filter=Prefix(property=asset_vid.as_property_ref("path"), value=root_path),
    limit=500,
)
print(f"Prefix filter call took: {time.time() - start_time:.3f} seconds")
_summarize("sub_tree_nodes_prefix", sub_tree_nodes, asset_vid)

Prefix filter call took: 0.158 seconds
sub_tree_nodes_prefix (count=90):
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  VAL-PH-23-GC  name='Gas Compression System'  description='1st 2nd 3rd stage gas compression'
  VAL-PH-23-UT  name='Utilities System'  description='General utilities and support'
  VAL-PH-23-WT  name='Water Treatment System'  description='Seawater and chemical injection'
  626221d1-e8f3-4761-97bd-239e013c7f75  name='Module Support Structure'  description='Compressor module support steel'
  6b0eb178-2c4f-4fca-901b-2c59bf812582  name='Instrument Equipment Room'  description='Instrument equipment room enclosure'
  a4b91d72-0d1b-4902-a8d4-98dfcbeabf70  name='1st Stage Safety Relief Valve'  description='Pressure safety valve 1st stage discharge'
  cd5759fc-0fa0-4ce3-9831-47529722f738  name='2nd Stage Suction ESDV'  description='Emergency shutdown valve 2nd stage suction'
  0192bf8a-c9a6-4f47-b491-208e6c11e351  name='1st Sta

## 6) Graph traversals

Use server-side relation traversal (`from_`, `through`, `direction`) to avoid client-side N+1 fetch patterns.

In [32]:
asset_eid = sample[0].external_id if sample else None
if not asset_eid:
    raise RuntimeError("Need at least one asset to run traversal demo")

ASSET_PROPS = ["name", "description", "parent", "tags"]
query = Query(
    with_={
        "asset": NodeResultSetExpression(
            filter=And(
                Equals(("node", "externalId"), value=asset_eid),
                Equals(("node", "space"), value={"parameter": "space"}),
            )
        ),
        "parent": NodeResultSetExpression(from_="asset", through=asset_vid.as_property_ref("parent"), direction="outwards"),
        "children": NodeResultSetExpression(from_="asset", through=asset_vid.as_property_ref("parent"), direction="inwards"),
    },
    select={
        "asset": Select([SourceSelector(asset_vid, ASSET_PROPS)]),
        "parent": Select([SourceSelector(asset_vid, ASSET_PROPS)]),
        "children": Select([SourceSelector(asset_vid, ASSET_PROPS)]),
    },
    parameters={"space": INST_SP},
)

res = run_query(client, query)
_summarize("asset", res["asset"], asset_vid)
_summarize("parent", res["parent"], asset_vid)
_summarize("children", res["children"], asset_vid)

nested_query = Query(
    with_={
        "asset": NodeResultSetExpression(
            filter=Nested(
                scope=asset_vid.as_property_ref("parent"),
                filter=ContainsAll(property=asset_vid.as_property_ref("tags"), values=["functional-location"]),
            ),
            limit=100,
        )
    },
    select={"asset": Select([SourceSelector(asset_vid, ["name", "description", "tags", "parent"])])},
)
nested_res = run_query(client, nested_query)
_summarize("nested_assets", nested_res["asset"], asset_vid)

asset (count=1):
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
parent (count=1):
  VAL-PH-23  name='Area 23 Gas Compression'  description='Gas compression area 23'
children (count=12):
  a86482db-1947-43e7-9781-67c6bc38ad09  name='MV Motor 1st Stage Compressor Drive'  description='Medium voltage motor for 1st stage compressor'
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  8e337b6f-be72-416e-bef0-3308975ae627  name='VFD 1st Stage Compressor'  description='Variable frequency drive for 1st stage compressor'
  a98260bd-c718-446c-901a-913809d4f095  name='Gas Turbine Generator B'  description='GE LM2500 gas turbine generator set B'
  df052853-f39c-405b-a275-33768b91c623  name='Gas Turbine Generator A'  description='GE LM2500 gas turbine generator set A'
  ccdafea7-5bf0-4d40-924e-3f786b44e9a3  name='UPS System Process Hall'  description='Uninterruptible power 

## 7) Cursor pagination

Always page large result sets with stable batch sizes and cursor continuation rather than attempting one-shot full reads.

In [33]:
def get_data(
    client: CogniteClient,
    query: Query,
    max_iterations: int | None = 100,
) -> tuple[dict[str, list[NodeListWithCursor | EdgeListWithCursor]], dict[str, str]]:
    if any(c for c in (query.cursors or {}).values()):
        print("Cursors already set in query, continuing retrieval.")

    collected_data: dict[str, list] = defaultdict(list)
    current_iteration = 0
    if max_iterations is None or max_iterations == -1:
        max_iterations = float("inf")

    res = None
    while current_iteration < max_iterations:
        res = run_sync(client, query)

        if res is None:
            if not collected_data:
                return {}, {}
            return collected_data, {}

        if all(not res.data[selection] for selection in res.data):
            return collected_data, {}

        for selection in res.data:
            collected_data[selection].extend(res.data[selection])

        query.cursors = res.cursors
        current_iteration += 1

    return collected_data, (res.cursors if res is not None else {})

# /sync all-assets retrieval in 1000-row pages (no business filter)
PAGE_SIZE = 1000
all_assets_query = Query(
    with_={
        "assets": NodeResultSetExpression(
            filter=And(
                Equals(["node", "space"], value={"parameter": "space"}),
                HasData(views=[asset_vid]),
            ),
            limit=PAGE_SIZE,
        )
    },
    select={"assets": Select([SourceSelector(asset_vid, ["name", "description", "tags"])])},
    parameters={"space": INST_SP},
)

all_assets = []
page = 0
while True:
    res = run_sync(client, all_assets_query)
    batch = res.data.get("assets", [])
    if not batch:
        break
    page += 1
    all_assets.extend(batch)
    print(f"Page {page}: fetched {len(batch)} rows (running total={len(all_assets)})")
    all_assets_query.cursors = res.cursors

_summarize("all_assets", all_assets, asset_vid)

# get_data helper demo
paged_query = Query(
    with_={
        "assets": NodeResultSetExpression(
            filter=And(
                Equals(["node", "space"], value=INST_SP),
                HasData(views=[asset_vid]),
            ),
            limit=50,
        ),
    },
    select={"assets": Select([SourceSelector(asset_vid, ["name", "description", "tags"])])},
)
collected, final_cursors = get_data(client, paged_query, max_iterations=5)
_summarize("assets (total collected)", collected.get("assets", []), asset_vid)
print("More data available." if final_cursors else "Result set fully drained.")

Page 1: fetched 1000 rows (running total=1000)
Page 2: fetched 203 rows (running total=1203)
all_assets (count=1203):
  cce41cc2-8716-45e2-ae75-102289a6854b  name='Top Drive System'  description=None
  37b24df6-c4c4-4ffc-90d3-349ba346966c  name='Drilling Derrick Assembly'  description=None
  1d338ea3-7841-43fc-afe6-702966cbbffc  name='Seawater Lift Pump B'  description='Centrifugal seawater lift pump train B'
  4d839ff3-3a8f-4258-81d5-08409cdf2475  name='Portable Gas Detector'  description='Portable multi-gas detector for confined space'
  a86482db-1947-43e7-9781-67c6bc38ad09  name='MV Motor 1st Stage Compressor Drive'  description='Medium voltage motor for 1st stage compressor'
  e3e89763-dd36-4823-9738-17d9a4fecd02  name='PA/GA System Process Area'  description='Public address and general alarm system'
  056df394-acc1-4b71-9897-0053e2aec3b0  name='Hydraulic Torque Wrench 3/4in'  description='Hydraulic torque wrench for flange bolting'
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Ele